# Notation automatique de commentaires Marmiton (GRU) — version corrigée

**Ce qui change par rapport à l'ancien notebook :**
1. Suppression des doublons **après** nettoyage, et **plus d'échantillonnage avec remise** (`sample(replace=True)`).
2. Séparation train / validation / test **en tout premier** (stratifiée).
3. Génération de commentaires artificiels **uniquement à partir du train**, et Word2Vec entraîné sur le train seul.
4. Vocabulaire construit sur le train seul (mots inconnus → `<unk>`).
5. GRU avec séquences « packées » (le modèle ne lit plus le padding).
6. Choix de l'époque sur la validation, **score final sur le test**, avec F1 macro et comparaison à une baseline.
7. Deux expériences : **avec** et **sans** augmentation, pour mesurer son effet réel.

In [ ]:
import re, json, random, time, copy
from collections import Counter

import numpy as np
import pandas as pd
import nltk, demoji
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

nltk.download('stopwords')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

## 1. Chargement

In [ ]:
# ATTENTION : dans l'ancien notebook le nom contenait un espace avant ".json". Adapte si besoin.
DATA_PATH = 'comments_final1 .json'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

rows = []
for recipe in raw:
    for c in recipe["comments"]:
        rows.append((recipe["_id"]["$oid"], c["comment"], c["appreciation"]))

df = pd.DataFrame(rows, columns=['id', 'text', 'score'])
print("Commentaires bruts :", df.shape)

## 2. Nettoyage (identique à avant), puis dédoublonnage

In [ ]:
STOP_WORDS = set(stopwords.words('french'))

def cleanPunc(sentence):
    cleaned = re.sub(r'[?|!|\'|"|#]', r'', sentence)
    cleaned = re.sub(r'[.|,|)|(|\|/]', r' ', cleaned)
    cleaned = cleaned.strip().replace("\n", " ")
    return cleaned

def removeStopWords(sentence):
    return ' '.join([w for w in sentence.split() if w.lower() not in STOP_WORDS])

def parse_score(x):
    if isinstance(x, str):
        x = x.replace('/5', '').strip()
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return np.nan

df['text'] = df['text'].astype(str).apply(cleanPunc)
df['text'] = df['text'].apply(removeStopWords)
df['text'] = df['text'].str.lower()
df['text'] = df['text'].apply(lambda x: re.sub(r'http\S+', '', re.sub(r'[^\w\s]|[\d]|[\r\n]', '', x)))
df['text'] = df['text'].apply(lambda x: demoji.replace(x, ""))
df['text'] = df['text'].apply(lambda x: re.sub(r'(\w)(\1{2,})', r'\1', x))   # caractères répétés
df['text'] = df['text'].str.strip()

df['score'] = df['score'].apply(parse_score)
df = df.dropna(subset=['score'])
df = df[df['score'].between(1, 5)]
df['score'] = df['score'].astype(int)

# commentaires vides, trop longs
df = df[df['text'].str.len() > 0]
df['token_size'] = df['text'].apply(lambda x: len(x.split()))
df = df[df['token_size'] < 200]

# DOUBLONS : on garde un seul exemplaire de chaque texte (après nettoyage !)
n_before = len(df)
df = df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
print(f"Après nettoyage : {n_before} -> {len(df)} commentaires uniques")

df['label'] = df['score'] - 1          # notes 1..5 -> classes 0..4
print(df['label'].value_counts().sort_index())

## 3. Séparation train / validation / test (AVANT toute génération)

In [ ]:
train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df['label'], random_state=SEED)
train_df, val_df, test_df = [d[['text', 'label']].reset_index(drop=True) for d in (train_df, val_df, test_df)]

print("Tailles train/val/test :", len(train_df), len(val_df), len(test_df))
print("\nRépartition du train (%) :")
print((train_df['label'].value_counts(normalize=True).sort_index() * 100).round(1))

# Sécurité : aucun texte ne doit être présent dans deux ensembles
assert not (set(train_df.text) & set(test_df.text))
assert not (set(train_df.text) & set(val_df.text))
assert not (set(val_df.text) & set(test_df.text))
print("\nOK : aucun commentaire en commun entre train, validation et test.")

## 4. Augmentation de données (sur le TRAIN uniquement)

Pour chaque commentaire d'une classe rare (notes 1, 2, 3 → classes 0, 1, 2), on crée une ou deux variantes :
environ 30 % des mots sont remplacés par l'un de leurs 3 plus proches voisins dans l'espace Word2Vec (choisi au hasard).
Word2Vec est entraîné **sur le train seul**.

In [ ]:
def augment_train(train_df, copies={0: 2, 1: 2, 2: 1}, replace_ratio=0.3, topn=3):
    w2v = Word2Vec([t.split() for t in train_df['text']], vector_size=100, window=5,
                   min_count=2, workers=1, seed=SEED)
    rng = random.Random(SEED)
    existing = set(train_df['text'])
    new_texts, new_labels = [], []
    for text, label in zip(train_df['text'], train_df['label']):
        if label not in copies:
            continue
        words = text.split()
        for _ in range(copies[label]):
            out = []
            for w in words:
                if w in w2v.wv and rng.random() < replace_ratio:
                    neighbours = [n for n, _ in w2v.wv.most_similar(w, topn=topn)]
                    out.append(rng.choice(neighbours))
                else:
                    out.append(w)
            new = ' '.join(out)
            if new not in existing:          # pas de doublon, ni de copie identique
                existing.add(new)
                new_texts.append(new)
                new_labels.append(label)
    return pd.DataFrame({'text': new_texts, 'label': new_labels})

# Aperçu
aug_preview = augment_train(train_df)
print("Commentaires générés par classe :")
print(aug_preview['label'].value_counts().sort_index())
for i in range(3):
    print("\nORIGINAL :", train_df[train_df.label == 0]['text'].iloc[i])
    print("GÉNÉRÉ   :", aug_preview[aug_preview.label == 0]['text'].iloc[i])

## 5. Vocabulaire, encodage et modèle

In [ ]:
MAX_LEN = 128

def build_vocab(texts, min_freq=2):
    cnt = Counter(w for t in texts for w in t.split())
    itos = ['<pad>', '<unk>'] + [w for w, c in cnt.items() if c >= min_freq]
    return {w: i for i, w in enumerate(itos)}

def encode(texts, vocab, max_len=MAX_LEN):
    seqs, lens = [], []
    for t in texts:
        ids = [vocab.get(w, 1) for w in t.split()][:max_len]
        if not ids:
            ids = [1]
        lens.append(len(ids))
        ids = ids + [0] * (max_len - len(ids))
        seqs.append(ids)
    return torch.tensor(seqs, dtype=torch.long), torch.tensor(lens, dtype=torch.long)

class RateGRU(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden=256, n_classes=5, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x, lens):
        e = self.emb(x)
        packed = pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)          # h : dernier état caché réel (sans le padding)
        return self.fc(self.drop(h[-1]))

def predict(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, l, y in loader:
            out = model(x.to(device), l)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    return np.array(preds), np.array(labels)

## 6. Expérience : entraînement, choix de l'époque sur la validation, score sur le test

In [ ]:
CLASS_NAMES = ['1', '2', '3', '4', '5']

def run_experiment(use_aug, epochs=10, batch_size=64, lr=1e-3):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

    tr = train_df.copy()
    if use_aug:
        tr = pd.concat([tr, augment_train(train_df)], ignore_index=True)
    print(f"\n=== {'AVEC' if use_aug else 'SANS'} augmentation : {len(tr)} exemples d'entraînement ===")
    print(tr['label'].value_counts().sort_index().to_dict())

    vocab = build_vocab(tr['text'])
    Xtr, Ltr = encode(tr['text'], vocab);       ytr = torch.tensor(tr['label'].values, dtype=torch.long)
    Xva, Lva = encode(val_df['text'], vocab);   yva = torch.tensor(val_df['label'].values, dtype=torch.long)
    Xte, Lte = encode(test_df['text'], vocab);  yte = torch.tensor(test_df['label'].values, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(Xtr, Ltr, ytr), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(Xva, Lva, yva), batch_size=256)
    test_loader  = DataLoader(TensorDataset(Xte, Lte, yte), batch_size=256)

    model = RateGRU(len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_f1, best_state, best_epoch = -1, None, -1
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for x, l, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x.to(device), l), y.to(device))
            loss.backward()
            optimizer.step()
            total += loss.item()
        vp, vl = predict(model, val_loader)
        val_f1 = f1_score(vl, vp, average='macro')
        print(f"Époque {epoch:2d} | loss train {total/len(train_loader):.4f} | "
              f"val acc {accuracy_score(vl, vp):.3f} | val F1 macro {val_f1:.3f}")
        if val_f1 > best_f1:
            best_f1, best_epoch, best_state = val_f1, epoch, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    tp, tl = predict(model, test_loader)
    result = {
        'augmentation': use_aug,
        'meilleure_epoque': best_epoch,
        'test_accuracy': accuracy_score(tl, tp),
        'test_f1_macro': f1_score(tl, tp, average='macro'),
    }
    print(f"\nMeilleure époque (validation) : {best_epoch}")
    print("\n--- RÉSULTATS SUR LE JEU DE TEST ---")
    print(classification_report(tl, tp, target_names=CLASS_NAMES, zero_division=0))
    print("Matrice de confusion :\n", confusion_matrix(tl, tp))
    return result, model, vocab

res_sans, model_sans, vocab_sans = run_experiment(use_aug=False)
res_avec, model_avec, vocab_avec = run_experiment(use_aug=True)

## 7. Comparaison finale et baseline

In [ ]:
# Baseline : toujours prédire la classe la plus fréquente du train
majority = train_df['label'].value_counts().idxmax()
base_pred = np.full(len(test_df), majority)
base_acc = accuracy_score(test_df['label'], base_pred)
base_f1 = f1_score(test_df['label'], base_pred, average='macro', zero_division=0)

summary = pd.DataFrame([
    {'modèle': f'Baseline (toujours la note {majority+1})', 'accuracy': base_acc, 'F1 macro': base_f1},
    {'modèle': 'GRU sans augmentation', 'accuracy': res_sans['test_accuracy'], 'F1 macro': res_sans['test_f1_macro']},
    {'modèle': 'GRU avec augmentation', 'accuracy': res_avec['test_accuracy'], 'F1 macro': res_avec['test_f1_macro']},
]).round(3)
print(summary.to_string(index=False))

**Pour le CV, retiens :** la ligne *F1 macro* (elle ne se laisse pas gonfler par la classe « 5 » très majoritaire)
et l'écart entre *sans* et *avec* augmentation. **Envoie-moi ce tableau final** et je rédige l'entrée du CV avec tes vrais chiffres,
quels qu'ils soient.

## 8. Comparaison GRU vs LSTM

Même données, même séparation, même vocabulaire, mêmes hyperparamètres (256 unités, dropout 0,5, Adam, 10 époques) :
**seul le type de cellule récurrente change**. Les résultats GRU de la section 6 (`res_sans`, `res_avec`) sont réutilisés :
ne redémarre pas le noyau avant d'avoir exécuté ces cellules (sinon, relance d'abord les sections 1 à 7).

In [ ]:
class RateRNN(nn.Module):
    """Même architecture que RateGRU, avec une cellule GRU ou LSTM au choix."""
    def __init__(self, vocab_size, rnn_type='gru', emb_dim=256, hidden=256, n_classes=5, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        rnn_cls = nn.GRU if rnn_type == 'gru' else nn.LSTM
        self.rnn = rnn_cls(emb_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x, lens):
        e = self.emb(x)
        packed = pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        if isinstance(h, tuple):      # LSTM renvoie (h_n, c_n) : on garde h_n
            h = h[0]
        return self.fc(self.drop(h[-1]))

def run_experiment_rnn(use_aug, rnn_type, seed=SEED, epochs=10, batch_size=64, lr=1e-3, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    tr = train_df.copy()
    if use_aug:
        tr = pd.concat([tr, augment_train(train_df)], ignore_index=True)
    print(f"\n=== {rnn_type.upper()} | {'AVEC' if use_aug else 'SANS'} augmentation | graine {seed} | {len(tr)} exemples ===")

    vocab = build_vocab(tr['text'])
    Xtr, Ltr = encode(tr['text'], vocab);       ytr = torch.tensor(tr['label'].values, dtype=torch.long)
    Xva, Lva = encode(val_df['text'], vocab);   yva = torch.tensor(val_df['label'].values, dtype=torch.long)
    Xte, Lte = encode(test_df['text'], vocab);  yte = torch.tensor(test_df['label'].values, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(Xtr, Ltr, ytr), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(Xva, Lva, yva), batch_size=256)
    test_loader  = DataLoader(TensorDataset(Xte, Lte, yte), batch_size=256)

    model = RateRNN(len(vocab), rnn_type=rnn_type).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_f1, best_state, best_epoch = -1, None, -1
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for x, l, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x.to(device), l), y.to(device))
            loss.backward()
            optimizer.step()
            total += loss.item()
        vp, vl = predict(model, val_loader)
        val_f1 = f1_score(vl, vp, average='macro')
        if verbose:
            print(f"Époque {epoch:2d} | loss train {total/len(train_loader):.4f} | "
                  f"val acc {accuracy_score(vl, vp):.3f} | val F1 macro {val_f1:.3f}")
        if val_f1 > best_f1:
            best_f1, best_epoch, best_state = val_f1, epoch, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    tp, tl = predict(model, test_loader)
    print(f"Meilleure époque (validation) : {best_epoch}")
    print(classification_report(tl, tp, target_names=CLASS_NAMES, zero_division=0))
    print("Matrice de confusion :\n", confusion_matrix(tl, tp))
    return {'test_accuracy': accuracy_score(tl, tp), 'test_f1_macro': f1_score(tl, tp, average='macro'),
            'meilleure_epoque': best_epoch}

### Entraînement des LSTM (le plus long : 2 entraînements)

In [ ]:
res_lstm_sans = run_experiment_rnn(use_aug=False, rnn_type='lstm')
res_lstm_avec = run_experiment_rnn(use_aug=True,  rnn_type='lstm')

### Tableau final : baseline, GRU et LSTM

In [ ]:
majority = train_df['label'].value_counts().idxmax()
base_pred = np.full(len(test_df), majority)
base_acc = accuracy_score(test_df['label'], base_pred)
base_f1 = f1_score(test_df['label'], base_pred, average='macro', zero_division=0)

summary2 = pd.DataFrame([
    {'modèle': f'Baseline (toujours la note {majority+1})', 'accuracy': base_acc, 'F1 macro': base_f1},
    {'modèle': 'GRU sans augmentation',  'accuracy': res_sans['test_accuracy'],      'F1 macro': res_sans['test_f1_macro']},
    {'modèle': 'GRU avec augmentation',  'accuracy': res_avec['test_accuracy'],      'F1 macro': res_avec['test_f1_macro']},
    {'modèle': 'LSTM sans augmentation', 'accuracy': res_lstm_sans['test_accuracy'], 'F1 macro': res_lstm_sans['test_f1_macro']},
    {'modèle': 'LSTM avec augmentation', 'accuracy': res_lstm_avec['test_accuracy'], 'F1 macro': res_lstm_avec['test_f1_macro']},
]).round(3)
print(summary2.to_string(index=False))

**Envoie-moi ce tableau.** Attention : chaque résultat vient d'**une seule exécution** (une graine). Un écart de 2 à 3 points de F1 macro
entre deux modèles peut être du bruit. Si les deux modèles sont très proches, on pourra répéter avec 3 graines
(`run_experiment_rnn(..., seed=1)`, `seed=2`, `seed=3`) et comparer les moyennes.